In [ ]:
#SECTION 1 - LIBRARY
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langchain_community.embeddings import HuggingFaceEmbeddings
from IPython.display import Markdown, display   

In [ ]:
#SECTION 2 - API KEY
# Paste your Gemini API key below, or set GOOGLE_API_KEY in the environment
# (see .env.example at the repository root). A free key is available from
# Google AI Studio: https://aistudio.google.com/apikey
import os

GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "INSERT-YOUR-GEMINI-API-KEY-HERE")

if GOOGLE_API_KEY.startswith("INSERT"):
    print("No API key set. The retrieval sections below still run offline;")
    print("only the Gemini calls need a key.")
else:
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("Gemini API key loaded from the environment.")


In [ ]:
#SECTION 3 - LOAD PDF DOCUMENTS
# The four source papers are not committed: they are third-party research PDFs
# totalling ~8 MB. See data/papers/README.md for the titles and where to get
# them, then drop them into data/papers/.
from pathlib import Path

PAPERS = Path.cwd().parent / "data" / "papers"
pdf_files = sorted(str(p) for p in PAPERS.glob("*.pdf"))

if not pdf_files:
    raise FileNotFoundError(
        f"No PDFs in {PAPERS}. See data/papers/README.md for the four papers "
        f"this notebook indexes."
    )

documents = []
for path in pdf_files:
    loader = PyPDFLoader(path)
    documents.extend(loader.load())

print(f"Loaded {len(documents)} pages from {len(pdf_files)} PDFs")
for p in pdf_files:
    print(f"  - {Path(p).name}")


In [ ]:
#SECTION 4 - SPLIT DOCUMENTS INTO CHUNKS
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
split_docs = text_splitter.split_documents(documents)
print(f"Total chunks created: {len(split_docs)}")

In [ ]:
#SECTION 5 - CREATE EMBEDDING MODEL
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
#SECTION 6 - CREATE FAISS VECTOR DATABASE
vectorstore = FAISS.from_documents(
    documents=split_docs,
    embedding=embeddings
)
print("FAISS Vector Store Created Successfully")

In [ ]:
#SECTION 7 - CREATE RETRIEVER
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

In [ ]:
#SECTION 8 - CREATE RAG TOOL
@tool
def rag_search(query: str) -> str:
    """
    Search the vector database for relevant information.
    Returns retrieved content with references.
    """
    docs = retriever.invoke(query)
    retrieved_text = []
    references = []
    for i, doc in enumerate(docs):
        source = os.path.basename(
            doc.metadata.get("source", "Unknown")
        )
        # Human-readable page numbering
        page = doc.metadata.get("page", 0) + 1
        content = doc.page_content
        retrieved_text.append(
            f"""
DOCUMENT {i+1}
Source: {source}
Page: {page}
Content:
{content}
"""
        )
        references.append(
            f"- {source} — Page {page}"
        )
    final_response = f"""
RETRIEVED INFORMATION
==================================================
{chr(10).join(retrieved_text)}
REFERENCES
==================================================
{chr(10).join(references)}
"""
    return final_response

In [ ]:
#SECTION 9 - TEST RAG TOOL
test_result = rag_search.invoke(
    "What is Agentic RAG?"
)
print(test_result)

In [ ]:
#SECTION 10 - CREATE GEMINI LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

In [ ]:
#SECTION 11 - CREATE REACT AGENT
agent = create_react_agent(
    model=llm,
    tools=[rag_search],
    prompt="""
You are an AI Research Assistant specialized in:
- Agentic AI
- RAG
- ReAct
- LangGraph
- AI workflows
You MUST use the rag_search tool to answer questions.
Instructions:
1. Use retrieved information from the RAG tool
2. Give accurate and concise explanations
3. Do not make up information
4. ALWAYS include references at the end
5. Mention the PDF source name and page number
Format your answers EXACTLY like this:
<your explanation>
References:
- source.pdf — Page X
- source.pdf — Page Y
"""
)

In [ ]:
#SECTION 12 - ASK QUESTIONS TO THE AGENT
questions = [
    "What is ReAct?",
]

In [ ]:
#SECTION 13 - RUN QUESTIONS
for q in questions:
    print(f"QUESTION:\n{q}\n")
    response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": q
                }
            ]
        }
    )
    # Extract clean text only
    answer = response["messages"][-1].content
    # If content is returned as list/dict format
    if isinstance(answer, list):
        answer = answer[0]["text"]
    print("ANSWER:")
    # Proper markdown rendering
    display(Markdown(answer))